# Class 8 — pandas, continued: select, sort, group, create

In Class 7 we saw **how pandas thinks**: a table is a set of typed columns, operations apply to whole columns,
a boolean column (a *mask*) selects rows, every row has a label (the *index*), and `groupby` does
*split–apply–combine*.

Today we practise the four operations that you will need in **assignment #3**:

| question | pandas |
|---|---|
| Which rows? | masks: comparisons, `&` `\|` `~`, `.between`, `.isin`, string methods `.str...` |
| In which order? | `sort_values` (one or more columns), `nlargest` |
| What is the summary for each group? | `groupby` + `agg` |
| How do I create a new variable? | arithmetic on columns; `apply` with a function or a `lambda` |

- **Part 1** (guided): the four operations on the forest fires dataset of Class 7.
- **Part 2** (on your own): a real, messy dataset from INE on rural fires in Portugal (this was Part 3 of Class 7).

> Run the cells in order with **Shift+Enter**. If something breaks, *Restart* the kernel and *Run All*.

In [ ]:
import pandas as pd

URL = 'https://raw.githubusercontent.com/isa-ulisboa/greends-ipython/main/datasets/'
fires = pd.read_csv(URL + 'forestfires.csv')
print(pd.__version__)
fires.head()

Recall the columns: `X`, `Y` (cell in the park map), `month`, `day`, the FWI indices `FFMC`, `DMC`, `DC`, `ISI`,
`temp` (°C), `RH` (relative humidity, %), `wind` (km/h), `rain` (mm) and `area` (burned area, ha).

---
# Part 1 — Four operations

## 1.1 Selecting rows

A condition on a column gives a column of `True`/`False` (a **mask**), and `fires[mask]` keeps the rows where the mask is `True`:

In [ ]:
fires[fires['area'] > 100]

Conditions are combined with `&` (and), `|` (or) and `~` (not). **Each condition needs its own parentheses**,
because `&` is evaluated before `>` and `<`:

In [ ]:
fires[(fires['temp'] > 25) & (fires['RH'] < 30)]

A common error is to use the Python words `and`/`or`, which work for single values but not for whole columns:

In [ ]:
try:
    fires[(fires['temp'] > 25) and (fires['RH'] < 30)]
except ValueError as e:
    print('ValueError:', e)

`.between(a, b)` is a shortcut for `(column >= a) & (column <= b)`:

In [ ]:
fires[fires['temp'].between(20, 22)]

`.isin(list)` checks if each value is in a list, and `~` negates a mask:

In [ ]:
summer = ['jun', 'jul', 'aug', 'sep']
print(fires['month'].isin(summer).sum(), 'fires in summer')
print((~fires['month'].isin(summer)).sum(), 'fires in the other months')

**String methods.** The methods of Python strings (Class 1: `upper`, `startswith`, `replace`, `split`, ...)
are applied to a whole text column through `.str`. Many of them return a mask:

In [ ]:
fires['month'].str.upper().head()

In [ ]:
j_months = fires['month'].str.startswith('j')    # jan, jun, jul
fires.loc[j_months, 'month'].unique()

`.loc[mask, columns]` selects rows **and** columns at the same time. To count the rows that satisfy a
condition, use `mask.sum()` or `len(fires[mask])`.

**Exercise 1.1** — How many fires occurred in summer (`jun` to `sep`) with wind above 5 km/h and no rain?
Show only the columns `month`, `wind`, `rain` and `area` of those fires.

In [ ]:
mask = fires['month'].isin(summer) & (fires['wind'] > 5) & (fires['rain'] == 0)
print(mask.sum(), 'fires')
fires.loc[mask, ['month', 'wind', 'rain', 'area']]

## 1.2 Sorting

`sort_values` returns a **new** sorted table (`fires` itself does not change, unless you assign the result):

In [ ]:
fires.sort_values('area', ascending=False).head()

With a **list of columns**, rows are sorted by the first column, and rows with the same value in the first
column are sorted by the second one, and so on. `ascending` can also be a list, one value per column.
Here: months in alphabetical order, and, within each month, the largest fires first:

In [ ]:
fires.sort_values(['month', 'area'], ascending=[True, False]).head(10)

Text is sorted alphabetically (`apr`, `aug`, `dec`, ...), not in calendar order (see `reindex` in Class 7).
To get the *n* rows with the largest (or smallest) values, `nlargest` (or `nsmallest`) is a shortcut:

In [ ]:
fires.nlargest(5, 'area')

**Exercise 1.2** — Sort the fires by `day` and then by `temp`, highest temperature first. Then show the 3 hottest
fires of each weekday. (Hint: after sorting, `.groupby('day').head(3)` keeps the first 3 rows of each group.)

In [ ]:
by_day_temp = fires.sort_values(['day', 'temp'], ascending=[True, False])
by_day_temp.groupby('day').head(3)[['day', 'month', 'temp', 'area']]

## 1.3 Group and aggregate: `groupby`

The pattern is `table.groupby(KEY)[COLUMN].FUNCTION()`: **split** the rows by the values of `KEY`, **apply**
`FUNCTION` to `COLUMN` in each group, and **combine** the results in a new table, with `KEY` as the index:

In [ ]:
fires.groupby('month')['area'].sum()

Several functions at once, with `.agg` and a list of function names (`size` counts the rows of each group):

In [ ]:
fires.groupby('month')['area'].agg(['size', 'sum', 'mean', 'max'])

To summarise several columns and choose the names of the result, use *named aggregation*:
`new_name=(column, function)`.

In [ ]:
fires.groupby('month').agg(
    n_fires=('area', 'size'),
    mean_temp=('temp', 'mean'),
    total_area=('area', 'sum'),
)

The operations can be chained: **select** the rows, **group**, and **sort** the result.
For instance, the median area of the fires that burned something, by month:

In [ ]:
fires[fires['area'] > 0].groupby('month')['area'].median().sort_values(ascending=False)

With **two keys**, each group is a combination of values. `reset_index()` turns the keys back into ordinary columns:

In [ ]:
fires[fires['month'].isin(['aug', 'sep'])].groupby(['month', 'day']).size().reset_index(name='n_fires')

**Exercise 1.3** — For each weekday, compute the number of fires, the mean temperature and the total burned area,
and sort the result by total burned area (largest first). Then repeat, using only the fires with `area > 0`.

In [ ]:
summary = fires.groupby('day').agg(n_fires=('area', 'size'), mean_temp=('temp', 'mean'), total_area=('area', 'sum'))
display(summary.sort_values('total_area', ascending=False))
burned = fires[fires['area'] > 0]
summary = burned.groupby('day').agg(n_fires=('area', 'size'), mean_temp=('temp', 'mean'), total_area=('area', 'sum'))
display(summary.sort_values('total_area', ascending=False))

## 1.4 New variables: arithmetic, `apply` and `lambda`

When the new variable is a formula, use arithmetic on whole columns (Class 7). It is simple and fast:

In [ ]:
fires['area_km2'] = fires['area'] / 100
fires[['area', 'area_km2']].head()

When the rule needs an `if`/`else`, or is a function of your own, use `.apply(f)`: pandas calls the function `f`
on **each value** of the column, and returns a new column with the results:

In [ ]:
def size_class(a):
    if a == 0:
        return 'none'
    elif a < 10:
        return 'small'
    else:
        return 'large'

fires['size_class'] = fires['area'].apply(size_class)     # the function, without ()
fires['size_class'].value_counts()

A **lambda** is a short function without a name, written in one line: `lambda a: expression` is the same as
`def f(a): return expression`. It is convenient when the function is used only once, inside `apply`:

In [ ]:
fires['burned'] = fires['area'].apply(lambda a: 'yes' if a > 0 else 'no')
fires[['area', 'burned']].head()

A lambda contains a single expression. For rules with several cases (like `size_class`), a `def` is easier to read.

The function can also use a dictionary, for instance to translate codes into names:

In [ ]:
season = {'dec': 'winter', 'jan': 'winter', 'feb': 'winter', 'mar': 'spring', 'apr': 'spring', 'may': 'spring',
          'jun': 'summer', 'jul': 'summer', 'aug': 'summer', 'sep': 'autumn', 'oct': 'autumn', 'nov': 'autumn'}
fires['season'] = fires['month'].apply(lambda m: season[m])
fires[['month', 'season']].head()

**Applying a function to each row.** With `axis=1`, `apply` calls the function on **each row** (a Series), so the
function can use several columns of the row, by name:

In [ ]:
fires['hot_dry'] = fires.apply(lambda row: row['temp'] > 25 and row['RH'] < 30, axis=1)
fires['hot_dry'].sum()

This particular variable could also be computed with a mask, `(fires['temp'] > 25) & (fires['RH'] < 30)`, which is
much faster on large tables. Rule of thumb: use arithmetic and masks when you can, and `apply` when you need a
function of your own.

New variables are often used to **group**:

In [ ]:
fires.groupby('size_class').agg(n_fires=('area', 'size'), mean_temp=('temp', 'mean'), mean_RH=('RH', 'mean'))

**Exercise 1.4** — (a) Create a column `wind_class` with `'calm'` (wind below 2 km/h), `'moderate'` (2 to 5 km/h) and
`'strong'` (above 5 km/h), with `apply`. For each wind class, compute the number of fires and the mean burned area.

(b) Create a column `weekend` with `True` for `sat` and `sun`, with `apply` and a `lambda`. Is the proportion of fires
that burned something (`area > 0`) different on weekends? (Hint: the mean of a boolean column is a proportion.)

In [ ]:
def wind_class(w):
    if w < 2:
        return 'calm'
    elif w <= 5:
        return 'moderate'
    else:
        return 'strong'

fires['wind_class'] = fires['wind'].apply(wind_class)
display(fires.groupby('wind_class')['area'].agg(['size', 'mean']))

fires['weekend'] = fires['day'].apply(lambda d: d in ['sat', 'sun'])
print('weekend: ', (fires.loc[fires['weekend'], 'area'] > 0).mean())
print('weekdays:', (fires.loc[~fires['weekend'], 'area'] > 0).mean())

---
# Part 2 — On your own: rural fires in Portugal (INE)

The file `rural_fires.csv` was downloaded from [INE](https://www.ine.pt), the Portuguese national statistics institute.
It contains the **number of rural fires in 2023**, by geographic unit and by cause of fire. The geographic units follow
the NUTS classification (version 2024), and each one has a **hierarchical code**:

| code | level | examples |
|---|---|---|
| `PT` | country | Portugal |
| 1 character | NUTS I | `1` Continente, `2` Açores, `3` Madeira |
| 2 characters | NUTS II | `11` Norte, `19` Centro, `1A` Grande Lisboa, `1B` Península de Setúbal, `1C` Alentejo, `1D` Oeste e Vale do Tejo, `15` Algarve, `20`, `30` |
| 3 characters | NUTS III | `111` Alto Minho, `112` Cávado, ... |
| 7 characters | municipality | `1111601` Arcos de Valdevez |

The code of a municipality starts with the code of its NUTS II region (2 characters) and of its NUTS III region (3 characters).
For each unit there is the total number of fires and the number by cause: negligence, intentional, natural,
re-ignition, undetermined and not investigated.

Real data are rarely clean, and official statistics are usually prepared to be read by **people**, not by programs.
A typical workflow is: **look** at the raw file → **read** it → **clean** it → **select**, **sort**, **group**, **create**
(Part 1) → **report**.

Before using pandas, look at the raw file: read it as plain text (as in Class 6) and print the first 15 lines.

In [ ]:
import urllib.request

with urllib.request.urlopen(URL + 'rural_fires.csv') as f:
    lines = f.read().decode('utf-8').splitlines()
for i, line in enumerate(lines[:15]):
    print(i, line)

Things to notice:

- there are 11 lines of titles before the data, and notes at the end of the file;
- the separator is `;`, not `,`;
- every number is followed by ` &` (a flag added by INE), so pandas reads the numbers as **text**;
- the first column mixes a **code** and a **name** (`1111601: Arcos de Valdevez`);
- the rows mix different units: the country, NUTS regions and municipalities;
- the table is *wide*: the cause of fire is spread over six columns.

The following cell reads only the 347 data rows, and gives the columns short names:

In [ ]:
causes = ['negligence', 'intentional', 'natural', 'reignition', 'undetermined', 'not_investigated']
rf = pd.read_csv(URL + 'rural_fires.csv', sep=';', skiprows=11, nrows=347, header=None,
                 usecols=range(8), names=['place', 'total'] + causes)
rf.head()

The exercises go from basic to challenging. Do as many as you can.

## Cleaning

**2.1** Check `rf.dtypes`: the numbers are text (`object`, or `str` in pandas 3). Convert the column `total` and the
six columns of causes to integers. There are two ways to do it:

- with string methods on the whole column: `rf['total'].str.replace('&', '').str.strip().astype(int)`;
- with `apply` and a `lambda` that converts one value: `rf['total'].apply(lambda s: int(s.replace('&', '').strip()))`.

Use a `for` loop over the list `['total'] + causes` to convert all columns.

In [ ]:
for col in ['total'] + causes:
    rf[col] = rf[col].str.replace('&', '').str.strip().astype(int)
    # or: rf[col] = rf[col].apply(lambda s: int(s.replace('&', '').strip()))
rf.dtypes

**2.2** Split the column `place` into two new columns, `code` and `name`.
(Hint: `rf[['code', 'name']] = rf['place'].str.split(': ', n=1, expand=True)`; what do `n=1` and `expand=True` do?)

In [ ]:
rf[['code', 'name']] = rf['place'].str.split(': ', n=1, expand=True)
rf[['code', 'name', 'total']].head()

## Selecting rows

**2.3** Create a dataframe `mun` with the municipalities only (codes with 7 characters: `.str.len()`). How many are there?
Then add the columns `nuts2` and `nuts3` with the first 2 and 3 characters of the code (`.str[:2]`, `.str[:3]`).

Use `mun = rf[...].copy()`: `.copy()` makes `mun` an independent table, so that adding columns to `mun` does not
raise a warning about `rf` in older versions of pandas.

In [ ]:
mun = rf[rf['code'].str.len() == 7].copy()
mun['nuts2'] = mun['code'].str[:2]
mun['nuts3'] = mun['code'].str[:3]
print(len(mun), 'municipalities')
mun.head()

**2.4** Answer each question with a mask:

- (a) How many fires were there in Lisboa, Porto, Coimbra, Évora, Faro and Bragança? (`.isin`)
- (b) How many municipalities have a name that starts with `'Vila'`, and how many fires did they have in total?
- (c) How many municipalities had no fires at all?
- (d) Which municipalities of the Alentejo (`nuts2` equal to `'1C'`) had more than 20 fires?

In [ ]:
cities = ['Lisboa', 'Porto', 'Coimbra', 'Évora', 'Faro', 'Bragança']
display(mun.loc[mun['name'].isin(cities), ['name', 'total']])

vila = mun['name'].str.startswith('Vila')
print(vila.sum(), 'municipalities, with', mun.loc[vila, 'total'].sum(), 'fires')

print((mun['total'] == 0).sum(), 'municipalities without fires')

mun.loc[(mun['nuts2'] == '1C') & (mun['total'] > 20), ['name', 'total']]

## Sorting

**2.5** Which 10 municipalities had the most fires? Then sort `mun` by `nuts2` and, within each region, by `total`
(largest first), and show the 3 municipalities with the most fires in each NUTS II region.

In [ ]:
display(mun.nlargest(10, 'total')[['name', 'total']])
ranked = mun.sort_values(['nuts2', 'total'], ascending=[True, False])
ranked.groupby('nuts2').head(3)[['nuts2', 'name', 'total']]

## Validating and creating new variables

**2.6** *Validate the data*: does the sum of `total` over the municipalities equal the value for Portugal?
And, for each municipality, is `total` equal to the sum of the six causes? (Hint: `mun[causes].sum(axis=1)` sums each row.)

In [ ]:
print(mun['total'].sum(), rf.loc[rf['code'] == 'PT', 'total'].item())
print((mun[causes].sum(axis=1) == mun['total']).all())

**2.7** Add a column `pct_negligence` with the percentage of fires due to negligence (arithmetic on columns).
Which municipalities with at least 50 fires have the highest percentage?

In [ ]:
mun['pct_negligence'] = 100 * mun['negligence'] / mun['total']
mun[mun['total'] >= 50].sort_values('pct_negligence', ascending=False)[['name', 'total', 'pct_negligence']].head(10)

**2.8** Create two new variables with `apply`:

- `fire_class`, from `total`, with `'none'` (0 fires), `'few'` (1 to 9), `'some'` (10 to 49) and `'many'` (50 or more).
  How many municipalities are in each class?
- `main_cause`, the cause with the largest number of fires in each municipality. Use `apply` on rows (`axis=1`) with
  `lambda row: row[causes].idxmax()`: `row[causes]` is a Series with the six causes, and `.idxmax()` returns the
  **label** of its largest value. What is the result for the municipalities without fires? Fix it, so that they get `'none'`.

In [ ]:
def fire_class(n):
    if n == 0:
        return 'none'
    elif n < 10:
        return 'few'
    elif n < 50:
        return 'some'
    else:
        return 'many'

mun['fire_class'] = mun['total'].apply(fire_class)
display(mun['fire_class'].value_counts())

mun['main_cause'] = mun.apply(lambda row: row[causes].idxmax() if row['total'] > 0 else 'none', axis=1)
mun['main_cause'].value_counts()

## Grouping

**2.9** For each NUTS II region, compute the number of municipalities, the total number of fires, and the mean and the
maximum number of fires per municipality. Sort by total number of fires. Then count, for each region, how many
municipalities have each `main_cause` (group by two columns).

To show the region names instead of the codes, select the NUTS II rows of `rf` (codes with 2 characters, except `'PT'`)
and use `merge` (Class 7).

In [ ]:
per_region = mun.groupby('nuts2').agg(
    n_mun=('total', 'size'),
    total=('total', 'sum'),
    mean=('total', 'mean'),
    max=('total', 'max'),
).sort_values('total', ascending=False)

nuts2 = rf.loc[(rf['code'].str.len() == 2) & (rf['code'] != 'PT'), ['code', 'name']]
nuts2 = nuts2.rename(columns={'code': 'nuts2', 'name': 'region'})
display(per_region.reset_index().merge(nuts2, on='nuts2', how='left'))

mun.groupby(['nuts2', 'main_cause']).size().reset_index(name='n_mun')

## Challenge

**2.10** The table is *wide*: the cause of fire is a variable spread over six columns. Use `melt` to create a tidy table
`tidy` with columns `code`, `name`, `nuts2`, `cause`, `n_fires` (one row per municipality and cause). How many rows should it have?

In [ ]:
tidy = mun.melt(id_vars=['code', 'name', 'nuts2'], value_vars=causes, var_name='cause', value_name='n_fires')
print(len(tidy), '=', len(mun), 'x', len(causes))
tidy.head()

**2.11** With `groupby` on the tidy table, compute the number of fires per region and cause. Then, for the final report,
turn it into a *wide* table (regions × causes, with the region names) with the **percentage** of each cause in each region.
Which region has the largest share of intentional fires? Why does one region have only `NaN` values?

In [ ]:
per_cause = tidy.merge(nuts2, on='nuts2', how='left').groupby(['region', 'cause'])['n_fires'].sum().reset_index()
report = per_cause.pivot_table(index='region', columns='cause', values='n_fires')
report = 100 * report.div(report.sum(axis=1), axis=0)
report.round(1).sort_values('intentional', ascending=False)

**2.12** Save your tidy table to `rural_fires_2023_tidy.csv` with `.to_csv(..., index=False)`. Why `index=False`?

In [ ]:
tidy.to_csv('rural_fires_2023_tidy.csv', index=False)